# Head Discovery — Instruct Models (paper §3.3)

Discovers the cultural binding heads: per-head attention *binding* features
(option→item attention mass), L1-regularized logistic regression with
scenario-level `GroupKFold` cross-validation, head stability across the 5
outer folds, feature-type comparison via edge knockout, and leave-one-out
analysis of the best head set.

Rebuilt from the master instruct pipeline notebook (`pipeline_instruct.ipynb`),
cells 10 (data/model loading), 12 (attention extraction + binding CV),
13 (per-feature-type CV runs), 15-16 (feature-type comparison + LOO). Shared helpers (data building, prompt formatting, span
detection, CV utilities, edge-knockout hooks) are imported from `common/`.

Output: `./results/<model>_instruct/binding_cv_results.pkl`.

In [ ]:
MODEL_KEY = "mistral"  # one of {"mistral", "llama", "gemma2", "nemo"}

In [ ]:
import sys, os
# Locate the repo root (the directory containing common/), whatever the kernel cwd
_p = os.path.abspath(".")
REPO_ROOT = _p if os.path.isdir(os.path.join(_p, "common")) else os.path.abspath("..")
assert os.path.isdir(os.path.join(REPO_ROOT, "common")), (
    "Cannot locate the repo root: run this notebook from its own directory or the repo root")
sys.path.insert(0, REPO_ROOT)

import pickle
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy import stats

import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger("transformers.generation.utils").setLevel(logging.ERROR)

from common import config

CFG = config.init(MODEL_KEY, "instruct")
SEED = config.SEED
DATA_DIR = config.DATA_DIR
OUTPUT_DIR = config.OUTPUT_DIR
HF_TOKEN = config.HF_TOKEN

from common.text_parsers import extract_options
from common.instruct.data import load_n4, build_factorial_as_conditions
from common.instruct.prompts import (format_for_chat, find_option_token_ids,
                                     detect_spans, validate_spans)
from common.instruct.discovery import extract_attention_scores, run_outer_cv
from common.instruct.hooks import compute_logit_scores_edge

## Stage 1 — Load data and model

In [ ]:
# ── Load data ──
cultural_items, neutral_items = load_n4(DATA_DIR)
data = build_factorial_as_conditions(cultural_items, seed=SEED)
n_total = len(data['B_cult'])
print(f"  {n_total} examples, {len(set(data['scenarios']))} scenarios")

# ── Load model ──
print(f"\n  Loading {CFG['model_path']}...")
tokenizer = AutoTokenizer.from_pretrained(
    CFG['model_path'], trust_remote_code=True, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(
    CFG['model_path'], dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN,
    attn_implementation="eager",
)
model.eval()
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

option_tokens_raw = find_option_token_ids(tokenizer)
first_device = next(model.parameters()).device
option_tokens = {opt: torch.tensor(ids, device=first_device)
                 for opt, ids in option_tokens_raw.items()}

for opt, ids in option_tokens_raw.items():
    decoded = [tokenizer.decode([i]) for i in ids]
    print(f"  option '{opt}': {decoded}")

# ── Format texts ──
conditions = ['B_cult', 'B_unrel']
texts_fmt = {c: format_for_chat(data[c], tokenizer) for c in conditions}

# Register runtime singletons so shared helpers in common/ can access them
config.model = model
config.tokenizer = tokenizer
config.first_device = first_device

## Stage 2 — Attention binding CV (head discovery)

In [ ]:
# ── RUN BINDING CV ──
print("=" * 80)
print(f"STAGE 2: ATTENTION BINDING CV — {CFG['label']}")
print("=" * 80)

all_features = {c: [] for c in conditions}
valid_indices = []
scenarios_valid = []
skipped = 0

for i in range(n_total):
    item_cult = data['items_cult'][i]
    cond_info = {}
    for cond in conditions:
        q_text = data[cond][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        cond_info[cond] = {'question': q_text, 'opt_a': oa, 'opt_b': ob, 'item': item_cult}

    prompts, spans_all = {}, {}
    all_ok = True
    for cond in conditions:
        info = cond_info[cond]
        full_text = data[cond][i]
        if CFG["chat_format"] == "gemma":
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": " " + full_text}],
                tokenize=False, add_generation_prompt=True)
        else:
            prompt = tokenizer.apply_chat_template(
                [{"role": "user", "content": full_text}],
                tokenize=False, add_generation_prompt=True)
        enc = tokenizer(prompt, return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, info['question'], info['opt_a'], info['opt_b'],
                          info['item'], tokenizer, item_required=True)
        if sp is None:
            all_ok = False; break
        prompts[cond] = prompt
        spans_all[cond] = sp
        del enc
    if not all_ok:
        skipped += 1; continue
    if len(valid_indices) < 3:
        for cond in conditions:
            enc = tokenizer(prompts[cond], return_tensors="pt")
            validate_spans(enc["input_ids"][0], spans_all[cond], tokenizer,
                         label=f"{cond}: {cond_info[cond]['opt_a']} vs {cond_info[cond]['opt_b']}")
            del enc
    for cond in conditions:
        scores = extract_attention_scores(model, tokenizer, prompts[cond], spans_all[cond])
        all_features[cond].append(scores)
    valid_indices.append(i)
    scenarios_valid.append(data['scenarios'][i])
    if len(valid_indices) % 50 == 0:
        print(f"  Processed {len(valid_indices)}/{n_total} (skipped {skipped})")

print(f"\n  Total valid: {len(valid_indices)} / {n_total} (skipped {skipped})")

In [ ]:

# ── Run CV for each feature type ──
FEATURE_NAMES = ['bind_avg', 'bind_a_to_item', 'bind_b_to_item']
cv_results = {}
for feat in FEATURE_NAMES:
    folds, summary = run_outer_cv(all_features, scenarios_valid, feature_name=feat)
    cv_results[feat] = {'folds': folds, 'summary': summary}

# Save
with open(OUTPUT_DIR / "binding_cv_results.pkl", "wb") as f:
    pickle.dump(cv_results, f)
print(f"\n  Saved binding CV results to {OUTPUT_DIR}")

## Stage 3 — Feature-type comparison + leave-one-out

In [ ]:
# ================================================================
# FEATURE-TYPE COMPARISON (which head set has causal signal?)
# ================================================================

print("=" * 80)
print(f"STAGE 3: FEATURE-TYPE COMPARISON — {CFG['label']}")
print("=" * 80)

# ── Build positions for edge knockout ──
texts_fmt = {cond: format_for_chat(data[cond], tokenizer) for cond in conditions}
positions = {c: [] for c in conditions}

for c in conditions:
    for i in range(n_total):
        if i not in valid_indices:
            positions[c].append(None); continue
        q_text = data[c][i].split("\n\n")[0]
        oa, ob = extract_options(q_text)
        item = data['items_cult'][i]
        enc = tokenizer(texts_fmt[c][i], return_tensors="pt")
        ids = enc["input_ids"][0].tolist()
        sp = detect_spans(ids, q_text, oa, ob, item, tokenizer, item_required=True)
        if sp is None:
            positions[c].append(None)
        else:
            assoc_pos = data['assoc_pos'][i]
            B_tokens = sp['opt_a'] if assoc_pos == 'a' else sp['opt_b']
            A_tokens = sp['opt_b'] if assoc_pos == 'a' else sp['opt_a']
            positions[c].append({
                'item_tokens': sp['item'], 'B_tokens': B_tokens,
                'A_tokens': A_tokens, 'B_text': oa if assoc_pos == 'a' else ob,
            })
        del enc

# ── Get top stable heads per feature type ──
def _heads_to_dict(head_list):
    d = {}
    for l, h in head_list:
        d.setdefault(l, []).append(h)
    return d

def run_feature_type_comparison(model, tokenizer, data, texts_fmt,
                                 positions, option_tokens, compute_fn):
    """Test each feature type's stable heads with edge knockout."""
    conditions_local = ['B_cult', 'B_unrel']
    print("\n  Computing baseline (no KO)...")
    scores_base = {}
    for c in conditions_local:
        scores_base[c] = compute_fn(
            model, tokenizer, texts_fmt[c], data[c],
            {}, positions[c], 'B_to_item', option_tokens)
    delta_base = scores_base['B_cult'].mean() - scores_base['B_unrel'].mean()
    diffs_base = scores_base['B_cult'] - scores_base['B_unrel']
    print(f"  Baseline Δ(S) = {delta_base:.4f}")

    results = {}
    for feat in FEATURE_NAMES:
        summary = cv_results[feat]['summary']
        if not summary or not summary.get('stable_heads'):
            print(f"\n  {feat}: no stable heads, skipping"); continue
        # Take heads with ≥ 4/5 folds
        best = [(lh, cnt) for lh, cnt in summary['stable_heads'] if cnt >= 4]
        if not best:
            best = summary['stable_heads'][:3]
        head_list = [lh for lh, _ in best]
        heads = _heads_to_dict(head_list)
        print(f"\n  {feat}: KO on {[f'L{l}H{h}' for l,h in head_list]}...")
        scores_ko = {}
        for c in conditions_local:
            scores_ko[c] = compute_fn(
                model, tokenizer, texts_fmt[c], data[c],
                heads, positions[c], 'B_to_item', option_tokens)
        delta_ko = scores_ko['B_cult'].mean() - scores_ko['B_unrel'].mean()
        diffs_ko = scores_ko['B_cult'] - scores_ko['B_unrel']

        # ── Signed paired t-test (per-pair difference) ──
        t_stat, p_val = stats.ttest_rel(diffs_base, diffs_ko)
        reduction = (1 - delta_ko / delta_base) * 100 if abs(delta_base) > 1e-10 else 0

        print(f"    Δ(S) = {delta_ko:.4f}  (reduction: {reduction:.1f}%)")
        print(f"    t = {t_stat:.3f}, p = {p_val:.4f}  (SIGNED, no abs)")

        results[feat] = {
            'heads': head_list, 'delta_ko': delta_ko, 'delta_base': delta_base,
            'reduction': reduction, 't_stat': t_stat, 'p_val': p_val,
            'diffs_base': diffs_base, 'diffs_ko': diffs_ko,
        }

    return results



# ================================================================
# LEAVE-ONE-OUT on best head set
# ================================================================

def run_loo(model, tokenizer, data, texts_fmt, positions, option_tokens, compute_fn):
    """Leave-one-out: necessity + sufficiency of individual heads."""
    # Pick the best feature type's heads
    best_feat = min(comparison_results.keys(),
                    key=lambda k: comparison_results[k]['p_val'])
    HEADS = comparison_results[best_feat]['heads']
    print(f"\n  LOO on best set ({best_feat}): {[f'L{l}H{h}' for l,h in HEADS]}")

    conditions_local = ['B_cult', 'B_unrel']
    # Baseline
    scores_base = {}
    for c in conditions_local:
        scores_base[c] = compute_fn(
            model, tokenizer, texts_fmt[c], data[c],
            {}, positions[c], 'B_to_item', option_tokens)
    delta_base = scores_base['B_cult'].mean() - scores_base['B_unrel'].mean()
    diffs_base = scores_base['B_cult'] - scores_base['B_unrel']
    print(f"  Baseline Δ(S) = {delta_base:.4f}")

    # All heads KO
    heads_all = _heads_to_dict(HEADS)
    scores_all = {}
    for c in conditions_local:
        scores_all[c] = compute_fn(
            model, tokenizer, texts_fmt[c], data[c],
            heads_all, positions[c], 'B_to_item', option_tokens)
    delta_all = scores_all['B_cult'].mean() - scores_all['B_unrel'].mean()
    diffs_all = scores_all['B_cult'] - scores_all['B_unrel']
    t_all, p_all = stats.ttest_rel(diffs_base, diffs_all)
    red_all = (1 - delta_all / delta_base) * 100 if abs(delta_base) > 1e-10 else 0
    print(f"\n  ALL KO: Δ={delta_all:.4f} (red={red_all:.1f}%), t={t_all:.3f}, p={p_all:.4f}")

    # Single head KO (necessity)
    print(f"\n  SINGLE HEAD KO (necessity):")
    for l, h in HEADS:
        heads_single = {l: [h]}
        scores_s = {}
        for c in conditions_local:
            scores_s[c] = compute_fn(
                model, tokenizer, texts_fmt[c], data[c],
                heads_single, positions[c], 'B_to_item', option_tokens)
        delta_s = scores_s['B_cult'].mean() - scores_s['B_unrel'].mean()
        diffs_s = scores_s['B_cult'] - scores_s['B_unrel']
        t_s, p_s = stats.ttest_rel(diffs_base, diffs_s)
        red_s = (1 - delta_s / delta_base) * 100 if abs(delta_base) > 1e-10 else 0
        print(f"    L{l:02d}H{h:02d}: Δ={delta_s:.4f} (red={red_s:.1f}%), t={t_s:.3f}, p={p_s:.4f}")

    # Leave-one-out (sufficiency of group, necessity of individual)
    print(f"\n  LEAVE-ONE-OUT (drop one, KO rest):")
    for l_out, h_out in HEADS:
        remaining = [(l, h) for l, h in HEADS if (l, h) != (l_out, h_out)]
        heads_r = _heads_to_dict(remaining)
        scores_r = {}
        for c in conditions_local:
            scores_r[c] = compute_fn(
                model, tokenizer, texts_fmt[c], data[c],
                heads_r, positions[c], 'B_to_item', option_tokens)
        delta_r = scores_r['B_cult'].mean() - scores_r['B_unrel'].mean()
        diffs_r = scores_r['B_cult'] - scores_r['B_unrel']
        t_r, p_r = stats.ttest_rel(diffs_base, diffs_r)
        red_r = (1 - delta_r / delta_base) * 100 if abs(delta_base) > 1e-10 else 0
        print(f"    drop L{l_out:02d}H{h_out:02d}: Δ={delta_r:.4f} (red={red_r:.1f}%), "
              f"t={t_r:.3f}, p={p_r:.4f}")

    return {'heads': HEADS, 'delta_base': delta_base}


In [ ]:
comparison_results = run_feature_type_comparison(
    model, tokenizer, data, texts_fmt, positions, option_tokens,
    compute_logit_scores_edge)
loo_results = run_loo(
    model, tokenizer, data, texts_fmt, positions, option_tokens,
    compute_logit_scores_edge)